# Character naming across the rise of the novel

Three naming categories tagged per character by `scripts/analyze_social_networks.py`:

- **realistic** — first name appears in the Galbi parish-register set (Northumberland & Durham, 1530-1830, 3,383 names). Vernacular English given names.
- **classical** — name ends in `-us`, `-ia`, `-issa`, `-andra`, `-ander`, `-enes`, `-oles`, `-inda`, `-etta`, `-ina`. Captures the Lysander/Andromaque/Clarissa/Pamela tradition.
- **type** — opens with `the`/`a`/`an` or `Possessive's` (e.g. `the Doctor`, `Lady Booby's nephew`). Allegorical or role-only naming.

All three are coarse heuristics, not LLM-classified. The trend is robust because the per-text metric averages over full character lists. Caveat: parish data is regionally narrow (NE England) — *realistic* tracks 18C English vernacular naming, not realism in the literary sense.

Data: `data/social_network_analysis.csv` (1,152 texts, output of `analyze_social_networks.py`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 200

df = pd.read_csv('../data/social_network_analysis.csv')
print(f'{len(df):,} texts; {df["year"].min()}-{df["year"].max()}; mean cast = {df["n_chars"].mean():.1f}')
df.head(3)

## 1. Trend across 1500-1900

Per-text mean of `realistic_pct`, `classical_pct`, `type_pct` in 25-year bins.

In [ ]:
BIN = 25
MIN_N = 5
in_range = df[(df['year'] >= 1500) & (df['year'] < 1925)].copy()
in_range['bin'] = (in_range['year'] // BIN * BIN).astype(int)
trend = (in_range.groupby('bin')
         .agg(n=('id', 'count'),
              realistic=('realistic_pct', 'mean'),
              classical=('classical_pct', 'mean'),
              type_name=('type_pct', 'mean'),
              real_se=('realistic_pct', lambda s: s.std() / np.sqrt(len(s))),
              clas_se=('classical_pct', lambda s: s.std() / np.sqrt(len(s))),
              type_se=('type_pct', lambda s: s.std() / np.sqrt(len(s))))
         .query(f'n >= {MIN_N}'))
trend.round(1)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x = trend.index + BIN / 2
for col, se, color, label in [
    ('realistic', 'real_se', '#1f77b4', 'realistic (parish names)'),
    ('classical', 'clas_se', '#d62728', 'classical (-us/-ia/-issa)'),
    ('type_name', 'type_se', '#2ca02c', 'type ("the Doctor")'),
]:
    y = trend[col].values
    ax.plot(x, y, 'o-', color=color, linewidth=2.2, markersize=6, label=label)
    ax.fill_between(x, y - 1.96 * trend[se].values, y + 1.96 * trend[se].values,
                    color=color, alpha=0.13)
ax.set_xlabel(f'year ({BIN}-year bin centers)')
ax.set_ylabel('% of characters in text')
ax.set_title('Character naming, 1500-1900: the rise of the parish-register name')
ax.grid(alpha=0.3)
ax.legend(loc='best', fontsize=9)
ax.xaxis.set_major_locator(MultipleLocator(50))
fig.tight_layout()
fig.savefig('../figures/character_naming_trend.png', bbox_inches='tight')
plt.show()

Realistic names roughly double across the 18C inflection (~13% mid-17C → ~28% by 1800). Classical naming collapses from 12-15% to ~1-2% over the same window. Type names fade more gradually. The classical/realistic crossover is around 1700.

## 2. By form

Genre form is the strongest non-temporal predictor — romance vs novel show different naming regimes.

In [ ]:
form_rows = []
for _, r in df.iterrows():
    for f in str(r['form']).split('; '):
        form_rows.append({**r.to_dict(), 'form_tag': f.strip()})
fdf = pd.DataFrame(form_rows)
top_forms = fdf['form_tag'].value_counts().head(8).index.tolist()
by_form = (fdf[fdf['form_tag'].isin(top_forms)]
           .groupby('form_tag')
           .agg(n=('id', 'count'),
                realistic=('realistic_pct', 'mean'),
                classical=('classical_pct', 'mean'),
                type_name=('type_pct', 'mean'))
           .loc[top_forms].round(1))
by_form

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(by_form))
w = 0.27
ax.bar(x - w, by_form['realistic'], w, color='#1f77b4', label='realistic')
ax.bar(x,     by_form['classical'], w, color='#d62728', label='classical')
ax.bar(x + w, by_form['type_name'], w, color='#2ca02c', label='type')
ax.set_xticks(x)
ax.set_xticklabels([f'{f}\n(n={n})' for f, n in zip(by_form.index, by_form['n'])], fontsize=9)
ax.set_ylabel('% of characters')
ax.set_title('Character naming by form (top 8 form tags)')
ax.grid(alpha=0.3, axis='y')
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig('../figures/character_naming_by_form.png', bbox_inches='tight')
plt.show()

## 3. Cross with abstractness

Pull text-level abstractness from `abstraction.scores` and ask: do abstract texts use more typological / classical names?

In [ ]:
import clickhouse_connect
c = clickhouse_connect.get_client(host='localhost', port=8123, username='lltk', password='lltk')
ids = df['source'].tolist()
scores = c.query_df(
    "SELECT _id, `Abs-Conc.Median.median` AS raw FROM abstraction.scores WHERE _id IN %(ids)s",
    parameters={'ids': ids},
).drop_duplicates('_id')
scores['abs_score'] = -scores['raw']  # + = more abstract
merged = df.merge(scores[['_id', 'abs_score']], left_on='source', right_on='_id', how='left')
have_abs = merged['abs_score'].notna()
print(f'{have_abs.sum()}/{len(merged)} texts with text-level abstractness')
merged[['realistic_pct', 'classical_pct', 'type_pct', 'abs_score']].dropna().corr().round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (col, color, label) in zip(axes, [
    ('realistic_pct', '#1f77b4', '% realistic names'),
    ('classical_pct', '#d62728', '% classical names'),
    ('type_pct',      '#2ca02c', '% type names'),
]):
    sub = merged.dropna(subset=[col, 'abs_score'])
    ax.scatter(sub[col], sub['abs_score'], s=12, alpha=0.35, color=color)
    if len(sub) > 5:
        slope, intercept = np.polyfit(sub[col], sub['abs_score'], 1)
        xs = np.linspace(sub[col].min(), sub[col].max(), 50)
        ax.plot(xs, slope * xs + intercept, color='black', linewidth=1.4)
        r = sub[col].corr(sub['abs_score'])
        ax.text(0.04, 0.96, f'r = {r:+.3f}', transform=ax.transAxes,
                fontsize=10, va='top')
    ax.set_xlabel(label)
    ax.grid(alpha=0.3)
axes[0].set_ylabel('text abstractness (+ = more abstract)')
fig.suptitle('Naming style vs. text-level abstractness', fontsize=12, y=1.02)
fig.tight_layout()
fig.savefig('../figures/character_naming_vs_abstractness.png', bbox_inches='tight')
plt.show()

## 4. Texts at the extremes

Top realistic-name books and top classical-name books, to sanity-check the heuristic.

In [ ]:
cols = ['year', 'author', 'title', 'form', 'n_chars',
        'realistic_pct', 'classical_pct', 'type_pct']
min_chars = 5
filt = df[df['n_chars'] >= min_chars]
print('TOP REALISTIC')
print(filt.nlargest(15, 'realistic_pct')[cols].to_string(index=False))
print()
print('TOP CLASSICAL')
print(filt.nlargest(15, 'classical_pct')[cols].to_string(index=False))
print()
print('TOP TYPE')
print(filt.nlargest(15, 'type_pct')[cols].to_string(index=False))

In [ ]:
c.close()